This project uses high frequency SPY data to
* Calculate Order Flow Imbalance (OFI) for supply-demand pressure.
* Engineered an adaptive Kalman Filter state-space model parameterizing observation noise covariance as a function of incoming order flow intensity.
* Fit a self-exciting Hawkes process via Maximum Likelihood Estimation to quantify shock clustering and branching ratios ($\alpha/\beta$).

### Import libraries

In [13]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

### 1. Import Market Data (High-Frequency SPY)

In [14]:
df = yf.download("SPY", period="60d", interval="15m")

df.info()
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

close_col = "Adj Close" if "Adj Close" in df.columns else "Close"

[*********************100%***********************]  1 of 1 completed

<class 'pandas.DataFrame'>
DatetimeIndex: 1560 entries, 2026-06-08 09:30:00-04:00 to 2026-09-01 15:45:00-04:00
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, SPY)   1560 non-null   float64
 1   (High, SPY)    1560 non-null   float64
 2   (Low, SPY)     1560 non-null   float64
 3   (Open, SPY)    1560 non-null   float64
 4   (Volume, SPY)  1560 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 73.1 KB


### 2. Construct Proxy Order Flow Imbalance (OFI) & Mid-Price Returns
$$\text{OFI} = sign(P_t-P_{t-1})*\log(1+\text{Volume}).$$
$$\text{Log Return} = \log(P_t/P_{t-1}).$$

In [15]:
df["Mid_Price"] = (df["High"] + df["Low"]) / 2.0
df["Price_Change"] = df["Mid_Price"].diff()
df["OFI"] = np.sign(df["Price_Change"]) * np.log1p(df["Volume"])
df["Log_Return"] = np.log(df["Mid_Price"] / df["Mid_Price"].shift(1))
df.dropna(inplace=True)

### 3. Kalman Filter

In [16]:
class AdaptiveKalmanFilter:
    def __init__(self, A=1.0, C=1.0, Q=1e-5, R_base=1e-3):
        self.A = A        # State transition
        self.C = C        # Observation matrix
        self.Q = Q        # Process noise (true drift volatility)
        self.R_base = R_base # Base measurement/microstructure noise
        self.x = df["Mid_Price"].iloc[0] # Initial state estimate
        self.P = 1.0      # Error covariance

    def filter_series(self, prices, ofi_signals):
        filtered_states = []
        for y, ofi in zip(prices, ofi_signals):
            # Predict
            x_pred = self.A * self.x
            P_pred = (self.A ** 2) * self.P + self.Q
            
            # Adaptive Noise: High OFI confidence reduces observation noise R
            R_adaptive = self.R_base / (1.0 + np.abs(ofi))
            
            # Kalman Gain & Update
            K = P_pred * self.C / ((self.C ** 2) * P_pred + R_adaptive)
            self.x = x_pred + K * (y - self.C * x_pred)
            self.P = (1.0 - K * self.C) * P_pred
            
            filtered_states.append(self.x)
        return np.array(filtered_states)

kf = AdaptiveKalmanFilter()
df["Filtered_Price"] = kf.filter_series(df["Mid_Price"].values, df["OFI"].values)
df["Filtered_Return"] = np.log(df["Filtered_Price"] / df["Filtered_Price"].shift(1))

### 4. Hawkes Process Fitting via Maximum Likelihood Estimation (MLE)

#### Hawkes Process
The conditional intensity function that measures the rate an event occurs in $[t,t+dt]$ given the condition that events occurred at time $0<t_1<t_2<...$ is
$$\lambda(t) = \mu + \alpha\sum_{t_i<t} e^{-\beta(t-t_i)},$$
where

$\mu$ is the underlying Poisson process,

$\alpha$ is the self-excitation intensity,

$\beta$ is the decay rate.

The branching ratio $\alpha/\beta$ is the expected number of sequential shocks triggered by a single shock
$$\int_0^\infty \alpha e^{-\beta t}dt = \alpha/\beta.$$
If the branching ratio is more than $1$, each arrivial will have positive probability to have infinite descendants (unstable).

#### Detect market shocks (returns exceeding 2 std devs)

The shocks are the events we are trying to fit using Hawkes process.

In [17]:
shocks = np.where(np.abs(df["Log_Return"]) > 2 * df["Log_Return"].std())[0]
event_times = shocks.astype(float)

#### Log likelihood for Hawkes Process
Likelihood $L$ measures the probability arrival_times are as observed given that the process is a Hawkes process with $\mu,\alpha,\beta$ as parameters.

Here is the formula for probability if we devide time into intervals of length $\Delta t$.
$$L = \prod_{i=1}^n (\lambda(t_i)\Delta t)\cdot\prod_{\text{no events}}e^{-\lambda(t_j)\Delta t}\approx (\Delta t)^n\prod_{i=1}^n \lambda(t_i)\cdot exp(-\int_0^{T_{max}}\lambda(t)dt).$$
Let's drop the constant $(\Delta t)^n$, so
$$\log L = \sum \lambda(t_i) - \int_0^{T_{max}}\lambda(t)dt = \sum_i \lambda(t_i) - [\mu T_{max} + (\alpha/\beta)\sum_i (1 - exp[-\beta(T_{max}-t_i)])].$$

In [18]:
def hawkes_neg_log_likelihood(params, arrival_times, T_max):
    mu, alpha, beta = params
    if mu <= 0 or alpha <= 0 or beta <= 0 or alpha >= beta:
        return 1e9  # Penalty for non-stationarity
    
    n = len(arrival_times)
    log_lik = 0.0
    
    for i in range(n):
        t_i = arrival_times[i]
        # Intensity λ(t_i)
        r_i = np.sum(np.exp(-beta * (t_i - arrival_times[:i])))
        intensity = mu + alpha * r_i
        log_lik += np.log(max(intensity, 1e-8))
        
    # Compensator integral term
    integral = mu * T_max + (alpha / beta) * np.sum(1.0 - np.exp(-beta * (T_max - arrival_times)))
    return -(log_lik - integral)

#### Fit Hawkes parameters $\mu,\alpha,\beta$

In [19]:
init_params = [0.1, 0.2, 0.5]
res = minimize(hawkes_neg_log_likelihood, init_params, 
               args=(event_times, float(len(df))), method='L-BFGS-B',
               bounds=[(1e-3, None), (1e-3, None), (1e-3, None)])

mu_fit, alpha_fit, beta_fit = res.x

### 5. Results

In [20]:
print("=== QUANTITATIVE MICROSTRUCTURE MODEL RESULTS ===")
print(f"Kalman Filter Tracking Variance Reduction: {np.var(df['Filtered_Return'].dropna()):.6f} vs Raw Variance: {np.var(df['Log_Return'].dropna()):.6f}")
print(f"Hawkes Self-Excitation Intensity (alpha): {alpha_fit:.4f}")
print(f"Hawkes Decay Rate (beta): {beta_fit:.4f}")
print(f"Branching Ratio (alpha/beta < 1 for stability): {alpha_fit / beta_fit:.4f}")

=== QUANTITATIVE MICROSTRUCTURE MODEL RESULTS ===
Kalman Filter Tracking Variance Reduction: 0.000000 vs Raw Variance: 0.000002
Hawkes Self-Excitation Intensity (alpha): 0.1005
Hawkes Decay Rate (beta): 0.3186
Branching Ratio (alpha/beta < 1 for stability): 0.3155
